# 13: Then vs now (E6, part 1): current generators

Our detectors were trained on Tiny GenImage, whose generators date from 2021-2023. Here we test them, without retraining, on images from **2024-2026 generators** taken from the OpenFake test split (Livernoche et al., 2025; dataset v2): GPT Image 1.5 and 2, Midjourney 7, Nano Banana Pro, Flux.2, Z-Image, Seedream 5 and others. The real photographs come from DOCCI and ImageNet.

Images are used **as distributed**: no re-compression. OpenFake mixes PNG and JPEG files, so the format and JPEG quality of every image are recorded and reported. Each image gets the same 16-pixel-aligned 224 x 224 centre crop as our training data.

This notebook also exports a small set of original files to Drive for the WhatsApp test (notebook 14).

GPU runtime needed. About 15 GB is downloaded (three of the 13 test files); total time about 30 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q huggingface_hub

import io
import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from torchvision.models import resnet50
from transformers import CLIPVisionModelWithProjection

ROOT = Path("/content/drive/MyDrive/minor-project")
RESULTS = ROOT / "results"
DOWNLOADS = Path("/content/openfake")
OUT_DIR = ROOT / "processed_openfake"
WHATSAPP = ROOT / "whatsapp" / "original"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
SHARDS = [0, 1, 2]           # which of the 13 test files to download
PER_GENERATOR = 200          # at most this many images per generator
MIN_PER_GENERATOR = 50       # generators with fewer usable images are left out
PER_REAL_SOURCE = 300        # DOCCI and ImageNet each
WHATSAPP_FAKES = ["gpt-image-1.5", "gpt-image-2", "midjourney-7", "nano-banana-pro", "flux.2-klein-9b"]
WHATSAPP_PER_SOURCE = 10     # 10 per fake generator, 25 per real source -> 100 images
SIZE = 224
SEED = 42
CLIP_NAME = "openai/clip-vit-large-patch14"
BATCH_SIZE = 128
COLORS = {"M1 ResNet-50": "#2a78d6", "M2 CLIP": "#eb6834", "M3 NPR": "#1baf7a", "M4 M2+M3": "#eda100"}

## Download and choose images

Only the label columns are read first, to pick which rows to use; the images are read afterwards.

In [ ]:
DOWNLOADS.mkdir(exist_ok=True)
files, tables = {}, []
for k in SHARDS:
    name = f"core/test-{k:05d}-of-00013.parquet"
    files[k] = hf_hub_download("ComplexDataLab/OpenFake", name, repo_type="dataset", local_dir=DOWNLOADS)
    t = pq.ParquetFile(files[k]).read(columns=["label", "model", "release_date", "row_id"]).to_pandas()
    t["shard"] = k
    t["row"] = np.arange(len(t))
    tables.append(t)
    print(f"downloaded test file {k}: {len(t)} rows")
catalog = pd.concat(tables, ignore_index=True)

counts = catalog[catalog["label"] == "fake"]["model"].value_counts()
generators = counts[counts >= MIN_PER_GENERATOR * 1.3].index.tolist()
print("Generators with enough images:", generators)

# shuffle, then keep 1.5x the target per source so that images failing the size/colour checks can be replaced
catalog = catalog.sample(frac=1, random_state=SEED)
wanted = []
for source, group in catalog.groupby("model"):
    if source in ("docci", "imagenet"):
        wanted.append(group.head(int(PER_REAL_SOURCE * 1.5)))
    elif source in generators:
        wanted.append(group.head(int(PER_GENERATOR * 1.5)))
wanted = pd.concat(wanted)
print(f"{len(wanted)} candidate images")

## Helper functions (same as notebook 03)

In [ ]:
STD_LUMA = np.array([
    16, 11, 10, 16, 24, 40, 51, 61,
    12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56,
    14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77,
    24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101,
    72, 92, 95, 98, 112, 100, 103, 99,
])


def estimate_quality(img):
    luma = np.array(img.quantization[0])
    best_quality, best_error = 0, np.inf
    for quality in range(1, 101):
        scale = 5000 // quality if quality < 50 else 200 - 2 * quality
        table = np.clip((STD_LUMA * scale + 50) // 100, 1, 255)
        error = np.abs(table - luma).sum()
        if error < best_error:
            best_quality, best_error = quality, error
    return best_quality


def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))

## Read, check and crop the chosen images

Rules, as for our training data: skip grayscale images and images smaller than 224 pixels; convert RGBA to RGB.

In [ ]:
RAW = DOWNLOADS / "raw"
RAW.mkdir(exist_ok=True)
records, crops = [], []
for k, path in files.items():
    pf = pq.ParquetFile(path)
    rows_here = wanted[wanted["shard"] == k].set_index("row")
    start = 0
    for g in range(pf.num_row_groups):
        n = pf.metadata.row_group(g).num_rows
        needed = rows_here.index[(rows_here.index >= start) & (rows_here.index < start + n)]
        if len(needed):
            images = pf.read_row_group(g, columns=["image"])["image"].to_pylist()
            for row in needed:
                raw = images[row - start]["bytes"]
                img = Image.open(io.BytesIO(raw))
                info = rows_here.loc[row]
                if img.mode in ("L", "LA", "1", "I", "I;16") or min(img.size) < SIZE:
                    continue
                records.append({
                    "row_id": info["row_id"], "shard": k, "row": int(row),
                    "source": info["model"], "label": info["label"], "release_date": info["release_date"],
                    "format": img.format, "width": img.size[0], "height": img.size[1],
                    "jpeg_quality": estimate_quality(img) if img.format == "JPEG" else np.nan,
                })
                crops.append(np.asarray(center_crop_aligned(img.convert("RGB"))))
                if info["model"] in ["docci", "imagenet"] + WHATSAPP_FAKES:
                    ext = "jpg" if img.format == "JPEG" else img.format.lower()
                    (RAW / f"{info['row_id']}.{ext}").write_bytes(raw)
        start += n
    print(f"test file {k}: {len(records)} usable images so far")

meta = pd.DataFrame(records)
meta["y"] = (meta["label"] == "fake").astype(int)
crops = np.stack(crops)

# keep the target number per source, in the shuffled order
order = {rid: i for i, rid in enumerate(wanted["row_id"])}
meta["order"] = meta["row_id"].map(order)
keep = []
for source, group in meta.sort_values("order").groupby("source"):
    target = PER_REAL_SOURCE if source in ("docci", "imagenet") else PER_GENERATOR
    if source not in ("docci", "imagenet") and len(group) < MIN_PER_GENERATOR:
        continue
    keep.extend(group.index[:target])
meta = meta.loc[sorted(keep)].reset_index()
crops = crops[meta["index"].to_numpy()]
meta = meta.drop(columns=["index", "order"])
print(f"Final set: {len(meta)} images")

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
np.save(OUT_DIR / "images.npy", crops)
meta.to_csv(OUT_DIR / "meta.csv", index=False)

overview = meta.groupby("source").agg(
    n=("y", "size"),
    released=("release_date", lambda d: d.dropna().min() if d.notna().any() else "-"),
    png_share=("format", lambda f: (f == "PNG").mean()),
    jpeg_quality=("jpeg_quality", "median"),
    median_width=("width", "median"),
).sort_values("released")
display(overview.round(2))
overview.to_csv(RESULTS / "e6_openfake_overview.csv")

## Export the WhatsApp test set

The original files (as distributed, not cropped) of 25 DOCCI, 25 ImageNet and 10 images from each of five headline generators are copied to `whatsapp/original/` on Drive. Notebook 14 explains how to send them through WhatsApp.

In [ ]:
WHATSAPP.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)
chosen = []
for source in ["docci", "imagenet"] + WHATSAPP_FAKES:
    rows = meta[meta["source"] == source]
    n = 25 if source in ("docci", "imagenet") else WHATSAPP_PER_SOURCE
    chosen.append(rows.sample(n=min(n, len(rows)), random_state=SEED))
chosen = pd.concat(chosen)

names = []
for i, row in enumerate(chosen.itertuples()):
    ext = "jpg" if row.format == "JPEG" else row.format.lower()
    name = f"img{i:03d}.{ext}"
    shutil.copy(RAW / f"{row.row_id}.{ext}", WHATSAPP / name)
    names.append(name)
chosen = chosen.assign(file=names)
chosen.to_csv(ROOT / "whatsapp" / "original_meta.csv", index=False)
shutil.make_archive(str(ROOT / "whatsapp" / "original"), "zip", WHATSAPP)
print(f"Exported {len(chosen)} files to {WHATSAPP} (and original.zip)")
display(chosen["source"].value_counts())

## Load the detectors (as in notebook 11)

In [ ]:
class NPRNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet50(weights=None)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False),
            base.bn1, base.relu, base.maxpool,
        )
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, 1)

    def forward(self, x):
        rebuilt = F.interpolate(F.interpolate(x, scale_factor=0.5, mode="nearest"), scale_factor=2, mode="nearest")
        npr = x - rebuilt
        x = self.stem(npr * 2.0 / 3.0)
        x = self.layer2(self.layer1(x))
        return self.fc(torch.flatten(self.pool(x), 1))


m1 = resnet50(weights=None)
m1.fc = nn.Linear(m1.fc.in_features, 1)
m1.load_state_dict(torch.load(ROOT / "checkpoints" / "m1_resnet50.pt", map_location=device))
m1 = m1.to(device).eval()

m3 = NPRNet()
m3.load_state_dict(torch.load(ROOT / "checkpoints" / "m3_npr.pt", map_location=device))
m3 = m3.to(device).eval()

dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
CLIP_SIZE = clip.config.image_size

best_C = json.load(open(RESULTS / "m2_clip.json"))["best_C"]
train_feats = np.load(ROOT / "features" / f"{CLIP_NAME.split('/')[-1]}_train.npy")
train_y = (pd.read_csv(ROOT / "processed" / "train_meta.csv")["label"] == "fake").astype(int).to_numpy()
clf = LogisticRegression(C=best_C, max_iter=5000).fit(train_feats, train_y)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def score_all(images):
    probs = {"M1 ResNet-50": [], "M2 CLIP": [], "M3 NPR": []}
    for start in range(0, len(images), BATCH_SIZE):
        x = torch.from_numpy(np.array(images[start:start + BATCH_SIZE])).to(device).permute(0, 3, 1, 2).float() / 255
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            imagenet_x = (x - IMAGENET_MEAN) / IMAGENET_STD
            probs["M1 ResNet-50"].append(torch.sigmoid(m1(imagenet_x).squeeze(1).float()).cpu().numpy())
            probs["M3 NPR"].append(torch.sigmoid(m3(imagenet_x).squeeze(1).float()).cpu().numpy())
        clip_x = x if x.shape[-1] == CLIP_SIZE else F.interpolate(x, size=CLIP_SIZE, mode="bicubic", align_corners=False)
        emb = clip(pixel_values=((clip_x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float()
        probs["M2 CLIP"].append(F.normalize(emb, dim=1).cpu().numpy())
    probs = {k: np.concatenate(v) for k, v in probs.items()}
    probs["M2 CLIP"] = clf.predict_proba(probs["M2 CLIP"])[:, 1]
    probs["M4 M2+M3"] = (probs["M2 CLIP"] + probs["M3 NPR"]) / 2
    return probs


print("Detectors loaded")

## Results

In [ ]:
probs = score_all(crops)
predictions = meta.copy()
for model, p in probs.items():
    predictions[model] = p
predictions.to_csv(RESULTS / "e6_openfake_predictions.csv", index=False)

real = predictions[predictions["y"] == 0]
rows = []
for source, group in predictions.groupby("source"):
    row = {"source": source, "n": len(group), "released": overview.loc[source, "released"]}
    for model in COLORS:
        row[f"{model} correct"] = float(((group[model] >= 0.5) == group["y"]).mean())
        if group["y"].iloc[0] == 1:
            both = pd.concat([real, group])
            row[f"{model} AUC"] = float(roc_auc_score(both["y"], both[model]))
    rows.append(row)
summary = pd.DataFrame(rows).set_index("source")
summary["is_real"] = summary.index.isin(["docci", "imagenet"])
summary = summary.sort_values(["is_real", "released"], ascending=[False, True]).drop(columns="is_real")
summary.to_csv(RESULTS / "e6_openfake_summary.csv")

print("Correct rate (real sources: called real; generators: detected as generated):")
display(summary[[f"{m} correct" for m in COLORS]].round(3))
print("\nROC-AUC, real images (DOCCI + ImageNet) vs each generator:")
display(summary[[f"{m} AUC" for m in COLORS]].dropna().round(3))

fakes = predictions[predictions["y"] == 1]
overall = {m: {"real_correct": float((real[m] < 0.5).mean()), "fake_detected": float((fakes[m] >= 0.5).mean()),
               "auc": float(roc_auc_score(predictions["y"], predictions[m]))} for m in COLORS}
display(pd.DataFrame(overall).T.round(3))
json.dump(overall, open(RESULTS / "e6_openfake_overall.json", "w"), indent=2)

## Does the file format matter?

Some generators appear in OpenFake both as PNG and as JPEG files. Comparing the two within one generator separates the effect of the format from the effect of the generator.

In [ ]:
by_format = fakes.groupby(["source", "format"]).apply(
    lambda g: pd.Series({"n": len(g), **{m: (g[m] >= 0.5).mean() for m in COLORS}}), include_groups=False)
by_format = by_format[by_format["n"] >= 20]
mixed = [s for s in by_format.index.get_level_values(0).unique() if len(by_format.loc[s]) > 1]
print("Detection rate by file format, for generators with both formats (at least 20 each):")
display(by_format.loc[mixed].round(3) if mixed else "none")
by_format.to_csv(RESULTS / "e6_openfake_by_format.csv")

## Chart

In [ ]:
sources = summary.index.tolist()
fig, ax = plt.subplots(figsize=(max(12, 1.1 * len(sources)), 5))
width = 0.2
x = np.arange(len(sources))
for k, (model, color) in enumerate(COLORS.items()):
    ax.bar(x + (k - 1.5) * width, summary[f"{model} correct"] * 100, width - 0.02, color=color, label=model)
labels = [f"{s}\n({summary.loc[s, 'released']})" if s not in ("docci", "imagenet") else f"{s}\n(real)" for s in sources]
ax.set_xticks(x, labels, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("correct (%)")
ax.set_ylim(0, 105)
ax.axhline(50, color="#999999", linewidth=1, linestyle="--")
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, 1.13), ncol=4, frameon=False)
ax.set_title("Detectors trained on 2021-2023 generators, tested on OpenFake", pad=32)
plt.tight_layout()
plt.savefig(RESULTS / "e6_openfake.png", dpi=120)
plt.show()